# Speech Commands Keyword Spotting

<p align="right">
Run Time: ~20 minutes with training included / ~15 minutes with training skipped
</p>

This notebook walks through the complete pipeline to train, quantize, convert, and
benchmark a DS-CNN model on the **Speech Commands** dataset for Akida 1 hardware.

The Speech Commands dataset has 10 classes (yes, no, up, down, left, right, on, off, stop, go) plus silence and unknown.

The pipeline follows the standard Akida workflow:
1. Train a float model
2. Post-training quantization (PTQ)
3. Quantization-aware training (QAT) fine-tuning
4. Conversion to Akida `.fbz` format
5. Hardware evaluation and benchmarking

In [1]:
import os
import numpy as np
import tensorflow as tf

from tf_keras.utils import set_random_seed

from cnn2snn import load_quantized_model

DATA_PATH = './data/sc10'
CONFIG_PATH = './configs/training_cfg.yml'
MODELS_DIR = './models_notebook/'
os.makedirs(MODELS_DIR, exist_ok=True)

RUN_FLOAT_TRAINING = True
RUN_QAT_TRAINING = True

# Must be called before any TF ops to make GPU ops (conv backward passes,
# bilinear resize, etc.) deterministic. Has a small throughput cost.
tf.config.experimental.enable_op_determinism()

2026-07-21 05:56:50.156608: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 05:56:50.164112: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784638610.172085 4182996 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784638610.174346 4182996 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784638610.181090 4182996 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Dataset

The **Speech Commands** dataset is loaded via TensorFlow Datasets (`speech_commands`).
On the first run, TFDS will automatically download and prepare the dataset to
`DATA_PATH`. Subsequent runs read from the local cache.

The dataset is split 85/10/5 (train/val/test). Samples are preprocessed to extract MFCC features and delivered as uint8 values (0–255).
Training applies data augmentation through time shifting, time masking and frequency masking (see `speech_commands_data_loader.py`).

To pre-download without training, run:
```bash
python -c "import tensorflow_datasets as tfds; tfds.load('speech_commands', data_dir='./data/sc10')"
```

In [2]:
import yaml

from speech_commands_data_loader import compute_mfcc_range, get_datasets

with open(CONFIG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

SEED = cfg['seed']
set_random_seed(SEED)

data_transform = compute_mfcc_range(data_dir=DATA_PATH)
train_ds, test_ds, val_ds = get_datasets(
    data_dir=DATA_PATH,
    batch_size=cfg["batch_size"],
    data_transform=data_transform,
    aug_enabled=cfg.get("aug_enabled", False),
    aug_time_shift_max_ms=cfg.get("aug_time_shift_max_ms", 100),
    aug_freq_mask_param=cfg.get("aug_freq_mask_param", 2),
    aug_time_mask_param=cfg.get("aug_time_mask_param", 10),
    shuffle_seed=cfg.get("seed"),
    aug_seed=cfg.get("seed"),
)

2026-07-21 05:57:04.917953: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-07-21 05:57:05.155702: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2026-07-21 05:57:07.080357: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


MFCC range [0.5–99.5 pct]: -51.4311 – 14.2340


## Model

Akida 1.0 (AKD1500) supports a well-defined set of layer operations. The reference DS-CNN
model as defined in MLPerf Tiny requires a small number of targeted changes to run entirely on-chip. Each change
is described below:

**1. Input convolution kernel shape: `(10, 4)` → `(5, 5)`**  
Akida 1.0's input convolution layer requires a **square kernel** of size 1, 3, 5, or 7.
The original `(10, 4)` kernel violates both constraints. A `(5, 5)` kernel is a practical
replacement — it covers a similar receptive field area and works well empirically on this task.

**2. Depthwise + Pointwise → Fused Separable blocks**  
In the original model, a ReLU activation is applied after the depthwise convolution and again
after the pointwise convolution. Akida 1.0 does not support an activation between the depthwise
and pointwise steps of a separable block — the two must be **fused** (a single activation after
the pointwise layer only). The `separable_conv_block` helper from `akida_models` produces this
correct fused structure.

**3. Global Average Pooling position: after ReLU → before ReLU**  
In Akida 1.0, the Global Average Pooling operation must come **before** the final ReLU in the
last separable block, not after it. The `post_relu_gap=False` argument to `separable_conv_block`
places the pooling correctly.

**4. Add a `Rescaling` layer at the input**  
The Akida hardware operates on **uint8 inputs** (values 0–255). Adding a `Rescaling` layer as
the first operation in the model folds the [0, 255] → [0, 1] normalization directly into the
model weights, so no external pre-processing is needed at inference time. The hardware passes
raw uint8 data in and the on-chip rescaling is handled transparently.

**5. Remove the output `softmax`; train with `from_logits=True`**  
Softmax is not implemented as an on-chip operation in Akida 1.0. Removing it and training with
`SparseCategoricalCrossentropy(from_logits=True)` avoids spurious warnings at conversion time.
For inference, classification is simply the `argmax` of the raw output logits — equivalent in
result, since softmax is monotonic and does not change the argmax.

In [3]:
from speech_commands_model import build_ds_cnn

model = build_ds_cnn(
    filters=cfg["filters"],
    dropout_initial=cfg["dropout_initial"],
    dropout_final=cfg["dropout_final"],
    weight_decay=cfg["weight_decay"],
    num_sep_conv_blocks=cfg.get("num_sep_conv_blocks", 4),
    classifier_head=cfg.get("classifier_head", "dense"),
)
model.summary()

Model: "ds_cnn"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 49, 10, 1)]       0         
                                                                 
 rescaling (Rescaling)       (None, 49, 10, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 25, 5, 64)         1664      
                                                                 
 conv2d/BN (BatchNormalizat  (None, 25, 5, 64)         256       
 ion)                                                            
                                                                 
 conv2d/relu (ReLU)          (None, 25, 5, 64)         0         
                                                                 
 dropout (Dropout)           (None, 25, 5, 64)         0         
                                                            

## Float Training

The model is trained in full float32 precision for 30 epochs using the Adam
optimiser and sparse categorical cross-entropy loss (with `from_logits=True`,
since the model head outputs raw logits rather than softmax probabilities).

The learning rate follows a cosine decay schedule with linear warmup, starting at `1e-6`, warming up to the configured peak learning rate (`cfg["lr_float"]`) over the first 10% of training steps, then decaying back toward 0 by the final epoch. 

**Increasing Sparsity with Activity Regularization**: ReLU activations already produce some sparsity (any negative pre-activation becomes zero),
but we can encourage further sparsity by adding an **activity regularizer** directly to the
activation layers. This adds a small penalty to the loss function, incentivizing the model to drive more activations toward zero during
training.

We use **Hoyer-square regularization** on the ReLU outputs, which computes the ratio of the L1 and L2 norms.
Its implementation can be found at `regularizers_custom.py`.

Set `RUN_FLOAT_TRAINING = True` above to train from scratch. Otherwise, the
cell below loads a float model from the `pretrained_models` folder.

In [4]:
from speech_commands_train import train_speech_commands_float

if RUN_FLOAT_TRAINING:
    train_speech_commands_float(model=model, train_ds=train_ds,
                                val_ds=val_ds, config=cfg)
    model.save(
        MODELS_DIR + 'speech_commands.h5',
        include_optimizer=False)
    print('Float model saved.')
else:
    float_model_path = 'pretrained_models/speech_commands.h5'
    model = load_quantized_model(float_model_path)
    model.compile(metrics=['accuracy'])

Adding Activity Regularization to ReLU layers
Float model saved.


/home/guest/.conda/envs/ak2191/lib/python3.12/site-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [5]:
_, float_acc = model.evaluate(test_ds, verbose=1)
print(f'Float accuracy (test): {float_acc:.4f}')
_, float_acc = model.evaluate(val_ds, verbose=1)
print(f'Float accuracy (val): {float_acc:.4f}')

77/77 [==============================] - 1s 7ms/step - loss: 0.3301 - accuracy: 0.9141
Float accuracy (test): 0.9141
158/158 [==============================] - 1s 6ms/step - loss: 0.2115 - accuracy: 0.9545
Float accuracy (val): 0.9545


## Quantization

Post-training quantization (PTQ) via `cnn2snn.quantize` converts the model
to fixed-point arithmetic:
- **Input**: 8-bit (`-i 8`)
- **Weights**: 4-bit (`-w 4`)
- **Activations**: 4-bit (`-a 4`)

4-bit quantization must be used to be compatible with Akida 1 hardware. Note though that
the first layer (both its inputs and weights) can be 8-bit.

In [6]:
import cnn2snn

quantized_model = cnn2snn.quantize(
    model,
    input_weight_quantization=8,
    weight_quantization=4,
    activ_quantization=4)
print('Model quantized to i8/w4/a4.')

Model quantized to i8/w4/a4.


Quantizing a model after training like this is referred to as Post-Training
Quantization (PTQ). It can slightly reduce accuracy (especially at 4-bits as
here) because the model was trained with continuous weights but is now 
evaluated with discrete values.

In [7]:
quantized_model.compile(metrics=['accuracy'])
_, ptq_acc = quantized_model.evaluate(test_ds, verbose=1)
print(f'PTQ accuracy (test): {ptq_acc:.4f}')
_, ptq_acc = quantized_model.evaluate(val_ds, verbose=1)
print(f'PTQ accuracy (val): {ptq_acc:.4f}')

77/77 [==============================] - 1s 5ms/step - loss: 0.1588 - accuracy: 0.8098
PTQ accuracy (test): 0.8098
158/158 [==============================] - 1s 4ms/step - loss: 0.1588 - accuracy: 0.9083
PTQ accuracy (val): 0.9083


## Quantization-Aware Training (QAT)

We can run Quantization Aware Training (QAT) to recover most of the drop in 
accuracy. QAT fine-tunes the quantized model for a few epochs (here, 25) at a
reduced learning rate. Note that, although it can sound intimidating,
QAT with BrainChip's quantization tools is no more complex than simply sending
the quantized model back through the same training pipeline that was used to
prepare the float model in the first place.

Here too `Hoyer-square` regulrization is applied to the model's activations.

Set `RUN_QAT_TRAINING = True` above to run QAT locally. Otherwise, the
cell below loads a QAT model from the `pretrained_models` folder.

In [8]:
from speech_commands_train import train_speech_commands_qat

if RUN_QAT_TRAINING:
    # We refetch the dataset, only to ensure reproducibility against the non-notebook pipeline.
    # This resets the shuffle seed on the training data
    train_speech_commands_qat(model=model, train_ds=train_ds,
                              val_ds=val_ds, config=cfg)
    quantized_model.save(
        MODELS_DIR + 'speech_commands_qat.h5',
        include_optimizer=False)
    print('QAT model saved.')
else:
    qat_model_path = 'pretrained_models/speech_commands_qat.h5'
    quantized_model = load_quantized_model(qat_model_path)
    quantized_model.compile(metrics=['accuracy'])

Adding Activity Regularization to QuantizedReLU layers
QAT model saved.


In [9]:
_, qat_acc = quantized_model.evaluate(test_ds, verbose=1)
print(f'QAT accuracy (test): {qat_acc:.4f}')
_, qat_acc = quantized_model.evaluate(val_ds, verbose=1)
print(f'QAT accuracy (val): {qat_acc:.4f}')

77/77 [==============================] - 0s 5ms/step - loss: 0.1588 - accuracy: 0.8098
QAT accuracy (test): 0.8098
158/158 [==============================] - 1s 4ms/step - loss: 0.1588 - accuracy: 0.9083
QAT accuracy (val): 0.9083


## Conversion to Akida Format

`cnn2snn.convert` compiles the quantized Keras model into an Akida `.fbz`
model that can be loaded and executed directly on AKD1500 hardware.
The converter verifies hardware compatibility and maps each layer to its
corresponding Akida primitive.

In [10]:
akida_model = cnn2snn.convert(quantized_model)

akida_model_path = os.path.join(MODELS_DIR, 'speech_commands_qat.fbz')
akida_model.save(akida_model_path)
print(f'Akida model saved to {akida_model_path}')
akida_model.summary()

Akida model saved to ./models_notebook/speech_commands_qat.fbz
                Model Summary                 
______________________________________________
Input shape  Output shape  Sequences  Layers
[49, 10, 1]  [1, 1, 12]    1          6     
______________________________________________

______________________________________________________________
Layer (type)                    Output shape  Kernel shape  

================ SW/conv2d-conv2d_1 (Software) ===============

conv2d (InputConv.)             [25, 5, 64]   (5, 5, 1, 64) 
______________________________________________________________
separable_conv2d (Sep.Conv.)    [25, 5, 64]   (3, 3, 64, 1) 
______________________________________________________________
                                              (1, 1, 64, 64)
______________________________________________________________
separable_conv2d_1 (Sep.Conv.)  [25, 5, 64]   (3, 3, 64, 1) 
______________________________________________________________
                    

## Evaluation of Akida Model

We now run evaluation through the Akida model, to check that accuracy is 
comparable to that obtained from the quantized tf_keras model. Here, we deliberately
use the software backend (the default, since we do not check for and map to
a connected hardware device): this delivers a
bit-accurate simulation of the results that will be obtained when running
the model on hardware.

In the accompanying [speech_commands_notebook_benchmark.ipynb](speech_commands_notebook_benchmark.ipynb)
the same evaluation is run using the hardware backend (if, of course, a hardware Akida
device is connected), allowing you to confirm that the results are identical.

### Run Evaluation on Akida

The Akida runtime cannot consume `tf.data.Dataset` objects directly, rather
it expects a 4D numpy array (n, h, w, c) in uint8 format. So we
iterate over validation batches manually.

The model output tensor has shape `(B, 1, 1, C)` which is squeezed to 
`(B, C)` before taking the class argmax.

In [13]:
from tqdm import tqdm

BATCH_SIZE = cfg["batch_size"]

labels_all = []
logits_all = []
for batch, label_batch in tqdm(val_ds, desc="Evaluating on Akida"):
    if not isinstance(batch, np.ndarray):
        batch = batch.numpy()

    logits_batch = akida_model.predict(batch, batch_size=BATCH_SIZE)

    logits_batch = logits_batch.squeeze(axis=(1, 2))
    labels_all.append(label_batch)
    logits_all.append(logits_batch)

labels_all = np.concatenate(labels_all)
logits_all = np.concatenate(logits_all)
preds = np.argmax(logits_all, axis=1)

akida_acc = float(np.mean(preds == np.array(labels_all)))
print(f'Akida accuracy (val): {akida_acc:.4f}')

Evaluating on Akida: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 158/158 [00:00<00:00, 202.62it/s]

Akida accuracy (val): 0.9083


### Activation Sparsity

Akida hardware skips computation for zero-valued activations, so activation
sparsity directly reduces both energy consumption and inference latency.
Below we measure per-layer sparsity on a 1024-sample calibration batch drawn
from the validation set.

In [19]:
import sys
sys.path.insert(0, '../../..') # for brainchip_utils

from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity
from speech_commands_data_loader import get_samples

NUM_SAMPLES = 1024

samples = get_samples(DATA_PATH, data_transform=data_transform, num_samples=NUM_SAMPLES)
sparsity_dict = compute_sparsity(akida_model, samples=samples)
pretty_print_sparsity(sparsity_dict)


Layer                  Sparsity
-------------------------------
conv2d                  93.39%
separable_conv2d        90.96%
separable_conv2d_1      92.36%
separable_conv2d_2      97.68%
separable_conv2d_3      60.00%
conv2d_1                 0.08%
-------------------------------
Mean                    72.41%


## Summary

The table below compares validation accuracy across the three model variants.
The goal is that QAT and Akida accuracy remain close to the float baseline.

In [20]:
print('Speech Commands results')
print('=' * 40)
print(f'  Float accuracy:     {float_acc * 100:.2f}%')
print(f'  QAT accuracy:       {qat_acc * 100:.2f}%')
print(f'  Akida accuracy:     {akida_acc * 100:.2f}%')

Speech Commands results
  Float accuracy:     95.45%
  QAT accuracy:       90.83%
  Akida accuracy:     90.83%
